## Apply Filtering Process

In [1]:
# Import the necessary libraries/packages
import numpy as np
import pandas as pd

In [2]:
# Upload data with the filtering features added:
resultdf = pd.read_parquet('data/datawithFF_roll10mins.parquet')

In [3]:
c = lambda n: f"{n}412"
phys = (resultdf[c("winddir")].between(150,240)
        & resultdf[c("windspeed")].between(2,12)
        & resultdf[c("power")].between(300,2200))
surv0 = phys & (resultdf[c("farm_z_wd")].abs() <= 3)

print("rows surviving physical+farm_z:", int(surv0.sum()))
print("circ_dif_mean on surv — describe:")
print(resultdf.loc[surv0, c("circ_dif_mean")].describe())
print("farm_z on surv — range:",
      resultdf.loc[surv0, c("farm_z_wd")].min(), resultdf.loc[surv0, c("farm_z_wd")].max())

rows surviving physical+farm_z: 38497
circ_dif_mean on surv — describe:
count    22528.000000
mean         2.683929
std         11.467571
min       -178.331023
25%          0.806359
50%          3.456027
75%          6.005719
max        178.678636
Name: circ_dif_mean412, dtype: float64
farm_z on surv — range: -2.8087232259603985 2.976753102091563


In [4]:
def circ_dev_from_median(x):
    """
    Absolute circular deviation of each value from the series' median, in degrees.
    Uses linear median (safe when values don't straddle the ±180 wrap),
    but wraps the deviation into [-180, 180] before taking abs.
    NaNs propagate as NaN (median() skips them).
    """
    med = x.median()                      # linear median in degrees, skipna
    dev = (x - med + 180) % 360 - 180     # wrap to [-180, 180]
    return dev.abs()


def _keep_unless_fails(keep, value, fails):
    """
    AND `keep` with a NaN-tolerant pass:
    a row is excluded ONLY if it has a valid (non-NaN) value that fails.
    Rows with NaN in `value` are left as-is (no evidence to exclude).
    """
    exclude = value.notna() & fails       # NaN -> False -> not excluded
    return keep & ~exclude


def filter_turbine(df, t,
                   wd_lo=150, wd_hi=240,
                   ws_lo=1, ws_hi=17,
                   pwr_lo=100, pwr_hi=2200,
                   z_thresh=2.0,
                   pct=0.9):
    """
    Returns a boolean mask (aligned to df.index) of rows to KEEP for turbine t.

    Missing-value policy:
      - Physical-range columns (winddir, windspeed, power): a NaN DROPS the row.
        These are the turbine's own core signals; a row missing them is unusable
        downstream regardless.
      - Screening-feature columns (farm_z, circ_dif_mean, circ_dif_std,
        ratio_std_comp): a NaN PASSES that step. A missing screening feature is
        "no evidence to exclude," not grounds for removal.

    Percentile cutoffs (pct) are computed on surviving rows, ignoring NaN.
    With pct=0.95 each percentile step removes the top ~5% of valid rows.
    """
    c = lambda name: f"{name}{t}"

    wd   = df[c("winddir")]
    ws   = df[c("windspeed")]
    pwr  = df[c("power")]
    fz   = df[c("farm_z_wd")]
    cdm  = df[c("circ_dif_mean")]
    cds  = df[c("circ_dif_std")]
    rsc  = df[c("ratio_std_comp")]

    # --- Step 1: physical range (absolute bounds) — NaN drops the row ---
    # .between() returns False for NaN, so NaN physical channels are excluded.
    keep = (
        wd.between(wd_lo, wd_hi) &
        ws.between(ws_lo, ws_hi) &
        pwr.between(pwr_lo, pwr_hi)
    )

    # --- Step 2: farm_z, fixed 2-SD two-sided — NaN passes ---
    keep = _keep_unless_fails(keep, fz, fz.abs() > z_thresh)

    # ---- percentile filters computed on SURVIVING rows only ----
    surv = keep.copy()

    # --- Step 3: circ_dif_mean, top (1-pct) by |value - median_j| — NaN passes ---
    cdm_dev = circ_dev_from_median(cdm)
    cutoff_cdm = cdm_dev[surv].quantile(pct)      # e.g. 0.95 -> cuts top 5%
    keep = _keep_unless_fails(keep, cdm_dev, cdm_dev > cutoff_cdm)

    # --- Step 4: circ_dif_std, top (1-pct) one-sided high — NaN passes ---
    cutoff_cds = cds[surv].quantile(pct)
    keep = _keep_unless_fails(keep, cds, cds > cutoff_cds)

    # --- Step 5: ratio_std_comp, top (1-pct) one-sided high — NaN passes ---
    cutoff_rsc = rsc[surv].quantile(pct)
    keep = _keep_unless_fails(keep, rsc, rsc > cutoff_rsc)

    return keep


def all_turbine_ids(df, stem="winddir"):
    return sorted(int(col[len(stem):]) for col in df.columns
                  if col.startswith(stem) and col[len(stem):].isdigit())


def apply_filters(df, verbose=True):
    masks, rows = {}, []
    for t in all_turbine_ids(df):
        m = filter_turbine(df, t)
        masks[t] = m

        # diagnostics: offset surfacing + retention + how much NaN-tolerance let through
        c = lambda name: f"{name}{t}"
        surv0 = (df[c("winddir")].between(150, 240)
                 & df[c("windspeed")].between(0, 17)
                 & df[c("power")].between(100, 2200)
                 & (df[c("farm_z_wd")].abs() <= 2.0))
        med = df.loc[surv0, c("circ_dif_mean")].median()
        cdm_nan_rate = df.loc[surv0, c("circ_dif_mean")].isna().mean()

        rows.append({
            "turbine": t,
            "surv_median_cdm": round(med, 2) if pd.notna(med) else np.nan,
            "n_surv_physz": int(surv0.sum()),
            "cdm_nan_rate": round(float(cdm_nan_rate), 3),
            "final_retained": int(m.sum()),
        })

    report = pd.DataFrame(rows).set_index("turbine").sort_index()
    if verbose:
        print(report)
    retained = report["final_retained"]
    return masks, retained, report

In [5]:
# 2. Run the per-turbine filters
masks, retained, report = apply_filters(resultdf)

# 3. Inspect how much survived per turbine — do this BEFORE pooling

print(retained)
print(f"\nTotal rows kept across turbines: {retained.sum()}")
print(f"Min / max per-turbine: {retained.min()} / {retained.max()}")

         surv_median_cdm  n_surv_physz  cdm_nan_rate  final_retained
turbine                                                             
401                -0.34         47210         0.416           41022
402                 4.16         25069         0.274           21092
403                -4.95         47939         0.395           42390
404                -3.17         46135         0.413           39922
405                 0.27         47898         0.429           42380
406                 0.54         60457         0.385           52577
407                -0.73         46692         0.433           41377
408                 5.62         44231         0.362           37391
409                 2.03         50372         0.421           43807
410                 3.97         49335         0.417           43274
411                -0.54         40820         0.434           35822
412                 3.25         49148         0.420           43104
413                 0.71         4

In [6]:
def screened_mask(df, t, wd_lo=150, wd_hi=240, ws_lo=1, ws_hi=17,
                  pwr_lo=100, pwr_hi=2200, z_thresh=2.0):
    """Rows that passed the spatial-uniformity screen with VALID features
    (i.e. were actually assessed, not NaN-passed)."""
    c = lambda n: f"{n}{t}"
    return (df[c("winddir")].between(wd_lo, wd_hi)
            & df[c("windspeed")].between(ws_lo, ws_hi)
            & df[c("power")].between(pwr_lo, pwr_hi)
            & (df[c("farm_z_wd")].abs() <= z_thresh)
            & df[c("circ_dif_mean")].notna()
            & df[c("circ_dif_std")].notna()
            & df[c("ratio_std_comp")].notna())


def extract_turbine(df, t, keep_mask):
    """Long-form rows for turbine t, preserving the ts index as a column.
    Adds `screened`: True if the row passed the uniformity screen on valid
    features, False if it was retained only because a screening feature was NaN."""
    c = lambda n: f"{n}{t}"
    cols = ["winddir", "windspeed", "windvane", "power", "nacdir",
            "targetwindvane", "ti", "state",
            "circ_dif_mean", "circ_dif_std", "ratio_std_comp", "farm_z_wd"]

    sub = df.loc[keep_mask, [c(x) for x in cols]].copy()
    sub.columns = cols                       # strip turbine suffix
    sub.insert(0, "ts", df.index[keep_mask])  # preserve timestamp
    sub.insert(1, "turbine", t)

    scr = screened_mask(df, t).reindex(df.index, fill_value=False)
    sub["screened"] = scr[keep_mask].values
    return sub


def build_long(df, masks):
    parts = [extract_turbine(df, t, masks[t]) for t in masks if masks[t].any()]
    long_df = pd.concat(parts, ignore_index=True)
    return long_df

In [7]:
long_df = build_long(resultdf, masks)

print(long_df.shape)
print(long_df["screened"].mean())            # overall screened fraction (~0.68 expected)
print(long_df.groupby("turbine")["screened"].mean().describe())

## Consider: adding a rolling average QoI for each turbine. Calculate BEFORE filtering, then can use that output later.

(2220390, 15)
0.5423029287647666
count    55.000000
mean      0.548844
std       0.053431
min       0.503513
25%       0.518790
50%       0.535264
75%       0.562197
max       0.840563
Name: screened, dtype: float64


In [8]:
UNS = len(long_df[np.abs(long_df["targetwindvane"])<0.1])
S = len(long_df[np.abs(long_df["targetwindvane"])>0.1])

UNS_post = len(long_df[(np.abs(long_df["targetwindvane"])<0.1)&(np.abs(long_df["screened"])==True)])
S_post = len(long_df[(np.abs(long_df["targetwindvane"])>0.1)&(np.abs(long_df["screened"])==True)])

print(UNS, S, UNS_post, S_post)

2092314 128042 1141792 62319


In [10]:
# Save the resulting dataframe as a .parquet file
# uncomment below
# long_df.to_parquet("data/filtered_data_roll10mins.parquet")